# Student Social Media and Mental Health: Task 1 and Task 2

**Source:** https://www.kaggle.com/datasets/shivasingh4945/student-social-media-and-mental-health-impact

**Selected target:** `Stress_Level`

This is a classification dataset because the target is categorical.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_CANDIDATES = [
    Path('data/student_mental_health/Student Social Media And Mental Health Impact.csv'),
    Path('lecture_tasks/data/student_mental_health/Student Social Media And Mental Health Impact.csv'),
    Path('../data/student_mental_health/Student Social Media And Mental Health Impact.csv'),
]
DATA_PATH = next(path for path in DATA_CANDIDATES if path.exists())
REPORT_DIR = DATA_PATH.parents[2] / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
TARGET = 'Stress_Level'

df = pd.read_csv(DATA_PATH)
df.head()

## Step 1a: Datatypes

In [ ]:
dtype_report = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'unique_values': df.nunique(dropna=False),
    'unique_percentage': (df.nunique(dropna=False) / len(df) * 100).round(2)
})
dtype_report

The categorical columns are expected to have object/string datatypes and the hours, age, unlock, and mental-health score columns are expected to be numeric. No values are changed in this preliminary stage.

## Step 1b: Shape and dataset size

In [ ]:
print('Shape:', df.shape)
print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df.head())

The dataset contains 5,000 rows and 13 columns. This is large enough for introductory classification analysis, but target-class counts must still be checked because a large total does not guarantee a balanced target.

## Step 1c: Missing values, duplicates, zeroes, and suspicious values

In [ ]:
print('Total missing values:', int(df.isna().sum().sum()))
display(df.isna().sum().sort_values(ascending=False))
print('Complete duplicate rows:', int(df.duplicated().sum()))

numeric_columns = df.select_dtypes(include='number').columns
zero_report = (df[numeric_columns] == 0).sum().sort_values(ascending=False)
display(zero_report.to_frame('zero_count'))

display(df[numeric_columns].describe().T)

The downloaded file contains two complete duplicate rows. This is a potential data-quality issue because duplicated observations can give some records extra influence during model training. They should be investigated before later preprocessing.

Zero values are not automatically errors. For example, zero physical activity or zero daily unlocks could be valid reported values. They should be judged using the meaning and plausible range of each variable.

## Step 1d: Target balance and categorical alerts

In [ ]:
target_counts = df[TARGET].value_counts(dropna=False)
target_report = target_counts.to_frame('count')
target_report['percentage'] = (target_report['count'] / len(df) * 100).round(2)
display(target_report)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x=TARGET, order=target_counts.index)
plt.title('Stress_Level distribution')
plt.xlabel('Stress level')
plt.ylabel('Number of students')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

categorical_columns = df.select_dtypes(include=['object', 'category']).columns
for column in categorical_columns:
    print(f'\n{column}: {df[column].nunique()} unique values')
    display(df[column].value_counts(dropna=False).head(20))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_columns = ['Gender', 'Academic_Level', 'Most_Used_Platform', 'Purpose_Of_Use']
for axis, column in zip(axes.flat, plot_columns):
    order = df[column].value_counts().index
    sns.countplot(data=df, x=column, order=order, ax=axis)
    axis.set_title(column)
    axis.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
df[numeric_columns].hist(figsize=(15, 10), bins=25)
plt.tight_layout()
plt.show()

## Preliminary issue list

- The target has four classes and is not perfectly balanced; the smallest class should be monitored during model evaluation.
- There are two complete duplicate rows.
- Numeric lifestyle variables need range and outlier checks.
- Categorical values need consistency checks for spelling and capitalization.
- High-cardinality columns should be identified before encoding.
- Missing values are absent in this downloaded version, but the result should be verified after any future data refresh.

## ydata-profiling report

In [ ]:
try:
    from ydata_profiling import ProfileReport
except ImportError:
    from data_profiling import ProfileReport

profile = ProfileReport(
    df,
    title='Student Social Media and Mental Health Preliminary EDA',
    explorative=True
)
profile.to_file(REPORT_DIR / 'student_mental_health_ydata_profile.html')